# BME1510 Assingment 5: Autoencoder Implementation
Important notes:
* This implementation of an autoencoder has two hidden layers in the Encoder and Decoder. Feel free to modify this code. For example, you can try adding or removing layers, adding dropout layers, increasing the number of training epochs etc.
* Be sure to apply standardization to the input feature data prior to running the autoencoder.
* As per assignment instructions, be sure to use a bottleneck dimensionality of k=5 for classifier performance comparison between feature susbets/representations.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import keras
from keras import layers

# Apply standardizarion to features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Input dimension (original number of features)
input_dim = X_train_scaled.shape[1]

# This is the size of our latent space. As per assignment instructions, use k=5 bottleneck dimensionality.
bottleneck_dim = 5

# This is our input layer
input_layer = keras.Input(shape=(input_dim,))
# "encoded" is the encoded representation of the input
encoded = layers.Dense(20, activation='relu')(input_layer)
encoded = layers.Dense(15, activation='relu')(encoded)
bottleneck = layers.Dense(bottleneck_dim, activation='relu')(encoded)

# "decoded" is the lossy reconstruction of the input
decoded = layers.Dense(15, activation='relu')(bottleneck)
decoded = layers.Dense(20, activation='relu')(decoded)
output_layer = layers.Dense(input_dim, activation='relu')(decoded)

# This model maps an input to its reconstruction
autoencoder = keras.Model(input_layer, output_layer)

# This model maps an input to its encoded representation
encoder = keras.Model(input_layer, bottleneck)

In [ ]:
# Compile model. Specify optimizer and loss function
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse'
)

# Print summary of autoencoder architecture.
autoencoder.summary()

In [ ]:
# Train autoencoder
history = autoencoder.fit(
    X_train_scaled, X_train_scaled,
    validation_data=(X_test_scaled, X_test_scaled),
    epochs=500,
    batch_size=32,
    verbose=1
)

# Compute final train and test reconstruction loss.
train_loss = autoencoder.evaluate(X_train_scaled, X_train_scaled, verbose=0)
test_loss = autoencoder.evaluate(X_test_scaled, X_test_scaled, verbose=0)

print(f"\nFinal Train Reconstruction Loss: {train_loss:.4f}")
print(f"Final Test Reconstruction Loss: {test_loss:.4f}")

In [ ]:
# Generate final latent features for train and test subsets, after training.
X_train_latent = encoder.predict(X_train_scaled)
X_test_latent = encoder.predict(X_test_scaled)

print("Latent feature shape:", X_train_latent.shape)